In [1]:
import pandas as pd
import os
import folium
import sqlite3

In [2]:
with sqlite3.connect("../data/processed/flight_data.db") as conn:
    # 2. Run a query and load directly into a DataFrame
    query = 'SELECT * from us_routes'
    routes_df = pd.read_sql_query(query, conn)
routes_df.head()

,airline_code,airline_id,source_airport_code,source_airport_id,destination_airport_code,destination_airport_id,codeshare,stops,equipment
0,3M,20710.0,FLL,3533.0,MCO,3878.0,NaN,0,SF3
1,3M,20710.0,FLL,3533.0,TPA,3646.0,NaN,0,SF3
2,3M,20710.0,JAX,3712.0,TPA,3646.0,NaN,0,SF3
3,3M,20710.0,MCO,3878.0,FLL,3533.0,NaN,0,SF3
4,3M,20710.0,MCO,3878.0,PNS,3564.0,NaN,0,SF3


In [3]:
with sqlite3.connect("../data/processed/flight_data.db") as conn:
    # 2. Run a query and load directly into a DataFrame
    query = 'SELECT * from large_airports'
    airports_df = pd.read_sql_query(query, conn)
airports_df.head()

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
0,16091,KABQ,large_airport,Albuquerque International Sunport,35.039976,-106.608925,5355.0,NaN,US,US-NM,Albuquerque,yes,KABQ,ABQ,KABQ,ABQ,http://www.abqsunport.com/,https://en.wikipedia.org/wiki/Albuquerque_Inte...,NaN
1,3371,KALB,large_airport,Albany International Airport,42.748299,-73.801697,285.0,NaN,US,US-NY,Albany,yes,KALB,ALB,KALB,ALB,http://www.albanyairport.com/,https://en.wikipedia.org/wiki/Albany_Internati...,NaN
2,3384,KATL,large_airport,Hartsfield Jackson Atlanta International Airport,33.636700,-84.428101,1026.0,NaN,US,US-GA,Atlanta,yes,KATL,ATL,KATL,ATL,https://www.atl.com,https://en.wikipedia.org/wiki/Hartsfield–Jacks...,NaN
3,3386,KAUS,large_airport,Austin Bergstrom International Airport,30.197535,-97.662015,542.0,NaN,US,US-TX,Austin,yes,KAUS,AUS,KAUS,AUS,http://www.ci.austin.tx.us/austinairport/,https://en.wikipedia.org/wiki/Austin-Bergstrom...,"BSM, KBSM, Bergstrom AFB"
4,3396,KBDL,large_airport,Bradley International Airport,41.938555,-72.688016,173.0,NaN,US,US-CT,Hartford,yes,KBDL,BDL,KBDL,BDL,http://www.bradleyairport.com/,https://en.wikipedia.org/wiki/Bradley_Internat...,"HFD, Hartford"


In [25]:
map = folium.Map(location=[44.9672, -103.7716], zoom_start=3)

In [26]:
# 2. Create feature groups
airports = folium.FeatureGroup(name="Airports")

# 4. Add groups to map
map.add_child(airports)

def apply_to_each_row(row):
    iata = row["iata_code"]
    lat = row["latitude_deg"]
    long = row["longitude_deg"]
    name = row["name"]
    if iata is not None and lat is not None and long is not None and name is not None:
        folium.Marker(
            location=[float(lat), float(long)],
            popup=iata,
            tooltip=name,
            icon=folium.Icon(color="blue", icon="info-sign")
).add_to(airports)

airports_df.apply(apply_to_each_row, axis=1)

from pyproj import Geod

routes = folium.FeatureGroup(name="Routes")

# 4. Add groups to map
map.add_child(routes)

# Define endpoints: [latitude, longitude]
boston = [42.3581, -71.0636]
sf = [37.7833, -122.4167]

# Initialize WGS84 geodesic model
geod = Geod(ellps="WGS84")

# pyproj uses (lon, lat) order for npts
# npts calculates N intermediate points between start and end
num_intermediate_points = 13
points = geod.npts(
    lon1=boston[1],
    lat1=boston[0],
    lon2=sf[1],
    lat2=sf[0],
    npts=num_intermediate_points,
)

# Combine start, intermediate points, and end into [lat, lon] format for Folium
coordinates = [boston] + [[lat, lon] for lon, lat in points] + [sf]

folium.PolyLine(
    locations=coordinates,
    color="#FF0000",
    weight=5,
    tooltip="From Boston to San Francisco",
).add_to(routes)
folium.LayerControl().add_to(map)
map